[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/18_feature_selection/90_exercises.ipynb)

# Bài tập — Chương 18: Feature Selection (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/18_feature_selection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_friedman1
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.linear_model import Lasso, LassoCV, LinearRegression, LogisticRegression, RidgeCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Phân biệt filter, wrapper, embedded — mỗi họ 2 ví dụ, 1 ưu điểm, 1 nhược điểm. Viết công thức VIF.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một nhà phân tích nói: "Lasso đã loại biến `income` (hệ số = 0) nên thu nhập không ảnh hưởng tới khả năng vỡ nợ". Phản biện.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Hồi quy feature $x_j$ theo các feature khác cho $R^2 = 0.9$. Tính VIF. Nếu $R^2 = 0.5$? Ngưỡng VIF = 10 tương ứng $R^2$ bằng bao nhiêu?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
vif_09 = None
vif_05 = None
r2_at_vif10 = None
### END CODE HERE ###

In [ ]:
check_close(vif_09, 10, message="VIF khi R² = 0.9")
check_close(vif_05, 2, message="VIF khi R² = 0.5")
check_close(r2_at_vif10, 0.9, message="R² tại VIF = 10")

**Câu 4.** Forward selection trên $d = 30$ feature, 5-fold CV, chọn tối đa $k = 6$ feature (không dừng sớm). Tổng số lần fit model? So với thử **mọi** tập con 6 feature?

In [ ]:
from math import comb

### START CODE HERE ### (≈ 2 dòng)
n_fits_forward = None
n_fits_exhaustive = None
### END CODE HERE ###
print(n_fits_forward, n_fits_exhaustive)

In [ ]:
check_equal(n_fits_forward, 5 * (30 + 29 + 28 + 27 + 26 + 25), "forward: 825 lần fit")
check_equal(n_fits_exhaustive, 5 * 593775, "vét cạn: gần 3 triệu lần fit")

## L3 — Code

**Bài 5.** Viết `backward_elimination(model, X, y, cv, n_keep)`: bắt đầu với mọi feature, mỗi bước bỏ feature mà **khi bỏ đi** điểm CV (R²) cao nhất, cho tới khi còn `n_keep` feature. Trả về list chỉ số còn lại.

In [ ]:
def backward_elimination(model, X, y, cv, n_keep):
    remaining = list(range(X.shape[1]))
    ### START CODE HERE ### (≈ 5 dòng)
    # lặp tới khi còn n_keep: với mỗi j trong remaining tính R² CV khi BỎ j; bỏ feature cho điểm cao nhất
    pass
    ### END CODE HERE ###
    return remaining

In [ ]:
X5, y5 = make_friedman1(n_samples=400, n_features=8, noise=0.5, random_state=3)
kept5 = backward_elimination(LinearRegression(), X5, y5, KFold(4, shuffle=True, random_state=0), n_keep=4)
print("giữ lại:", sorted(kept5))
check_equal(len(kept5), 4, "còn đúng 4 feature")
check_true(set(kept5) <= {0, 1, 2, 3, 4}, "không giữ feature nhiễu (x6–x8)")

**Bài 6.** Viết `vif_table(df)` trả về Series VIF cho mọi cột của DataFrame (hồi quy mỗi cột theo các cột còn lại bằng `LinearRegression`).

In [ ]:
def vif_table(df):
    X = df.to_numpy(float)
    out = {}
    ### START CODE HERE ### (≈ 4 dòng)
    # với mỗi cột j: hồi quy X[:, j] theo các cột còn lại (LinearRegression) → R² → out[col] = 1 / (1 − R²)
    pass
    ### END CODE HERE ###
    return pd.Series(out)

In [ ]:
z = rng.normal(size=500)
dfv = pd.DataFrame({"a": z + rng.normal(0, 0.2, 500), "b": z + rng.normal(0, 0.2, 500), "c": rng.normal(size=500)})
v = vif_table(dfv)
print(v.round(2))
check_greater(v["a"], 10, "a và b gần như trùng → VIF lớn")
check_less(v["c"], 1.1, "c độc lập → VIF ≈ 1")

## L4 — Debug

**Bài 7.** Đoạn sau chọn 10 feature tốt nhất bằng F-test trên **toàn bộ** dữ liệu (2 000 feature nhiễu, nhãn ngẫu nhiên) rồi mới CV, báo cáo ROC-AUC rất cao. Sửa: đưa `SelectKBest` vào Pipeline, lưu `honest_auc`.

In [ ]:
from sklearn.feature_selection import f_classif

Xn = rng.normal(size=(150, 2000))
yn = rng.integers(0, 2, 150)
### START CODE HERE ### (≈ 3 dòng)
selected = SelectKBest(f_classif, k=10).fit_transform(Xn, yn)                                    # 🐛 chọn trên toàn bộ dữ liệu
honest_auc = cross_val_score(LogisticRegression(max_iter=2000), selected, yn, cv=5, scoring="roc_auc").mean()
### END CODE HERE ###
print(f"ROC-AUC trung thực = {honest_auc:.3f}")

In [ ]:
check_less(abs(honest_auc - 0.5), 0.12, "nhãn ngẫu nhiên → ≈ 0.5")

## L5 — Phân tích

**Bài 8.** So sánh khả năng phát hiện feature thật của **F-test**, **mutual information** (`n_neighbors=10`) và **Lasso** trên Friedman (5 feature thật, 25 nhiễu, 1 500 mẫu — MI cần đủ dữ liệu để ước lượng ổn định): với mỗi phương pháp chọn top-5 (Lasso: 5 feature có |hệ số| lớn nhất ở `LassoCV`), đếm số feature thật trong top-5. Lưu dict `hits`. Phương pháp nào bỏ sót $x_3$? Vì sao?

In [ ]:
Xf, yf = make_friedman1(n_samples=1500, n_features=30, noise=1.0, random_state=4)
Xfs = StandardScaler().fit_transform(Xf)
true_set = {0, 1, 2, 3, 4}
### START CODE HERE ### (≈ 6 dòng)
top_f = None        # tập chỉ số top-5 theo F-test
top_mi = None       # top-5 theo mutual_info_regression(n_neighbors=10, random_state=0)
lasso_abs = None    # |hệ số| của LassoCV(cv=5, random_state=0) fit trên Xfs, từng feature
top_lasso = None    # top-5 theo lasso_abs — CHỈ tính feature có hệ số ≠ 0
hits = None         # dict tên → số feature thật trong top-5
missed_x3 = None    # list tên phương pháp không có chỉ số 2 trong top-5
### END CODE HERE ###
print("số feature thật trong top-5:", hits, "| bỏ sót x3:", missed_x3)

In [ ]:
check_equal(hits["mutual info"], 5, "MI tìm đủ 5 feature thật")
check_true("F-test" in missed_x3 and "Lasso" in missed_x3, "F-test và Lasso (tuyến tính) bỏ sót x3")
print(f"|hệ số Lasso| của x3 = {lasso_abs[2]:.3f} so với trung bình x1, x2, x4, x5 = {lasso_abs[[0, 1, 3, 4]].mean():.3f}")

✍️ Nhận xét:

## L6 — Thực tế

**Bài 9.** Trên `customer_churn.csv` (one-hot), thêm 30 cột nhiễu ngẫu nhiên. Dùng **L1-Logistic Regression** (`penalty="l1"`, `solver="liblinear"`). `C` tối ưu theo CV thường giữ khá nhiều cột nhiễu với hệ số nhỏ → chọn **`C` nhỏ nhất có ROC-AUC CV không kém quá 0.002 so với tốt nhất** (tinh thần "one-standard-error rule": ưu tiên model đơn giản khi điểm gần như bằng nhau). Báo cáo: số feature thật được giữ, số cột nhiễu bị giữ (`n_noise_kept`), ROC-AUC test (`test_auc`). Yêu cầu `n_noise_kept ≤ 5` và `test_auc ≥ 0.80`.

In [ ]:
churn = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
Xc = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
noise_cols = [f"noise_{i}" for i in range(30)]
Xc = pd.concat([Xc, pd.DataFrame(rng.normal(size=(len(Xc), 30)), columns=noise_cols)], axis=1)
yc = churn["churn"].to_numpy()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
### START CODE HERE ### (≈ 10 dòng)
grid = None          # GridSearchCV(Pipeline(StandardScaler, LogisticRegression(penalty="l1", solver="liblinear")), lưới C, roc_auc)
chosen_C = None      # C nhỏ nhất có mean_test_score ≥ best_score_ − 0.002
model9 = None        # pipeline train lại với chosen_C
kept = None          # tên các cột có hệ số ≠ 0
n_noise_kept, n_real_kept = None, None
test_auc = None
### END CODE HERE ###
print(f"C tốt nhất theo CV = {grid.best_params_['logisticregression__C']} | C được chọn = {chosen_C} | "
      f"giữ {n_real_kept} feature thật, {n_noise_kept}/30 nhiễu | ROC-AUC test = {test_auc:.4f}")

In [ ]:
check_less(n_noise_kept, 6, "≤ 5 cột nhiễu được giữ")
check_greater(test_auc, 0.80, "ROC-AUC test ≥ 0.80")

## L7 — Nâng cao

**Bài 10.** Cài đặt **stability selection** (Meinshausen & Bühlmann): lặp $B = 100$ lần, mỗi lần lấy ngẫu nhiên **một nửa** dữ liệu, fit `Lasso(alpha)` → ghi feature có hệ số ≠ 0. Trả về tần suất chọn mỗi feature với một **lưới alpha**: `freq[alpha_index, feature]`. Feature "ổn định" = tần suất max (qua các alpha) ≥ 0.8. So sánh với một lần `LassoCV` trên dữ liệu tuyến tính có 6 feature thật + 40 nhiễu.

In [ ]:
def stability_selection(X, y, alphas, B=100, seed=0):
    r = np.random.default_rng(seed)
    n = len(y)
    freq = np.zeros((len(alphas), X.shape[1]))
    ### START CODE HERE ### (≈ 6 dòng)
    # B lần: lấy ngẫu nhiên n//2 mẫu (không hoàn lại); với mỗi alpha fit Lasso, cộng (coef != 0) vào freq[alpha_index]
    # cuối cùng chia freq cho B
    pass
    ### END CODE HERE ###
    return freq

In [ ]:
rng10 = np.random.default_rng(5)
Xs10 = rng10.normal(size=(250, 46))
beta = np.r_[[3, -2.5, 2, -1.5, 1.2, 1.0], np.zeros(40)]
ys10 = Xs10 @ beta + rng10.normal(0, 2, 250)
alphas10 = np.logspace(-0.4, 0.2, 8)             # vùng alpha đủ lớn để Lasso thưa
freq = stability_selection(Xs10, ys10, alphas10, B=60)
stable = np.flatnonzero(freq.max(axis=0) >= 0.8)
lasso_once = np.flatnonzero(LassoCV(cv=5, random_state=0).fit(Xs10, ys10).coef_ != 0)
print(f"stability selection: {sorted(stable.tolist())} | LassoCV một lần: {len(lasso_once)} feature ({sum(i >= 6 for i in lasso_once)} nhiễu)")
check_true(set(stable.tolist()) == set(range(6)), "stability selection tìm đúng 6 feature thật, không nhiễu")
check_greater(sum(i >= 6 for i in lasso_once), 0, "LassoCV một lần thường giữ thêm feature nhiễu")